In [ ]:
import importlib.util
import os
import random
import subprocess
import sys
from pathlib import Path

REQUIRED_PACKAGES = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'sklearn': 'scikit-learn',
    'skimage': 'scikit-image',
    'cv2': 'opencv-contrib-python',
    'torch': 'torch',
    'torchvision': 'torchvision',
    'tqdm': 'tqdm',
    'PIL': 'pillow'
}

##################### Install any missing libraries before running the notebook ##################
missing = [pkg for module_name, pkg in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module_name) is None]
if missing:
    print('Installing missing packages:', ', '.join(missing))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from PIL import Image
from skimage.feature import hog
from sklearn.base import clone
from sklearn.cluster import MiniBatchKMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATASET_DIR = Path('./signature-verification-dataset')
IMAGE_SIZE = (128, 128)
TEST_SIZE = 0.25
CNN_EPOCHS = 12
CNN_EARLY_STOPPING_PATIENCE = 3
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
SIFT_CODEBOOK_SIZE = 64
SIFT_MAX_DESCRIPTORS_PER_IMAGE = 80
LEARNING_CURVE_POINTS = np.array([0.2, 0.4, 0.6, 0.8, 1.0])
VALID_SUFFIXES = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}

if not DATASET_DIR.exists():
    candidates = [p for p in Path('.').iterdir() if p.is_dir() and 'signature' in p.name.lower()]
    if len(candidates) == 1:
        DATASET_DIR = candidates[0]
    else:
        raise FileNotFoundError(
            f'Dataset folder not found. Set DATASET_DIR to your extracted Kaggle dataset path. Tried: {DATASET_DIR.resolve()}'
        )

##################### Get signer name and genuine/forged tag ##################
def extract_signer_and_status(path: Path):
    parts = [part for part in path.parts[:-1] if part.lower() not in {'train', 'test', 'val', 'valid', 'validation', 'images'}]
    folder = parts[-1]
    if folder.lower().endswith('_forg'):
        return folder[:-5], 'forged'
    return folder, 'genuine'

##################### Load and clean one signature image ##################
def load_signature_array(path: Path, image_size=IMAGE_SIZE):
    img = Image.open(path).convert('L').resize(image_size)
    arr = np.asarray(img, dtype=np.uint8)
    # Invert when background is darker than foreground to keep ink strokes bright and consistent.
    if arr.mean() < 127:
        arr = 255 - arr
    arr = cv2.GaussianBlur(arr, (3, 3), 0)
    _, arr = cv2.threshold(arr, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return arr

##################### Scan dataset files ##################
records = []
for path in sorted(DATASET_DIR.rglob('*')):
    if path.is_file() and path.suffix.lower() in VALID_SUFFIXES:
        signer, status = extract_signer_and_status(path)
        records.append({'path': path, 'signer': signer, 'status': status})

if not records:
    raise RuntimeError(f'No signature images were found under {DATASET_DIR.resolve()}')

##################### Build label table ##################
df = pd.DataFrame(records)
signer_counts = df['signer'].value_counts()
valid_signers = signer_counts[signer_counts >= 2].index
df = df[df['signer'].isin(valid_signers)].reset_index(drop=True)
label_names = sorted(df['signer'].unique())
label_to_idx = {label: idx for idx, label in enumerate(label_names)}
df['label'] = df['signer'].map(label_to_idx)

print('Dataset directory:', DATASET_DIR.resolve())
print('Total images:', len(df))
print('Unique signers:', df['signer'].nunique())
print(df.groupby('status').size().rename('count').to_frame())

sample_paths = df.sample(min(9, len(df)), random_state=SEED)
fig, axes = plt.subplots(3, 3, figsize=(12, 8))
for ax, (_, row) in zip(axes.ravel(), sample_paths.iterrows()):
    ax.imshow(load_signature_array(row['path']), cmap='gray')
    ax.set_title(f"{row['signer']} | {row['status']}")
    ax.axis('off')
for ax in axes.ravel()[len(sample_paths):]:
    ax.axis('off')
plt.suptitle('Sample Signature Images')
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
top_counts = df['signer'].value_counts().head(20)
sns.barplot(x=top_counts.index, y=top_counts.values, color='#1f77b4')
plt.xticks(rotation=90)
plt.title('Top 20 Signers by Number of Images')
plt.xlabel('Signer')
plt.ylabel('Image Count')
plt.tight_layout()
plt.show()

train_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    stratify=df['label'],
    random_state=SEED
)

X_train_imgs = [load_signature_array(path) for path in tqdm(train_df['path'], desc='Loading train images')]
X_test_imgs = [load_signature_array(path) for path in tqdm(test_df['path'], desc='Loading test images')]
y_train = train_df['label'].to_numpy()
y_test = test_df['label'].to_numpy()

##################### Compute classification scores ##################
def compute_metrics(y_true, y_pred):
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision,
        'recall': recall,
        'f1_score': f1
    }

##################### Plot train vs test accuracy ##################
def plot_learning_curve_from_fractions(model, X_train, y_train, X_test, y_test, fractions, title):
    train_scores, test_scores = [], []
    n = len(X_train)
    for frac in fractions:
        subset_size = max(2, int(n * frac))
        subset_X = X_train[:subset_size]
        subset_y = y_train[:subset_size]
        local_model = clone(model)
        local_model.fit(subset_X, subset_y)
        train_scores.append(local_model.score(subset_X, subset_y))
        test_scores.append(local_model.score(X_test, y_test))
    plt.figure(figsize=(8, 4))
    plt.plot(fractions, train_scores, marker='o', label='Train accuracy')
    plt.plot(fractions, test_scores, marker='s', label='Test accuracy')
    plt.title(title)
    plt.xlabel('Fraction of training data used')
    plt.ylabel('Accuracy')
    plt.ylim(0, 1.05)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()
    return train_scores, test_scores

print('\nExtracting HOG features...')
X_train_hog = np.array([
    hog(img / 255.0, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2), block_norm='L2-Hys')
    for img in tqdm(X_train_imgs, desc='HOG train')
])
X_test_hog = np.array([
    hog(img / 255.0, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2), block_norm='L2-Hys')
    for img in tqdm(X_test_imgs, desc='HOG test')
])

hog_model = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=2000, multi_class='auto', n_jobs=None))
])
hog_model.fit(X_train_hog, y_train)
hog_pred = hog_model.predict(X_test_hog)
hog_metrics = compute_metrics(y_test, hog_pred)
plot_learning_curve_from_fractions(hog_model, X_train_hog, y_train, X_test_hog, y_test, LEARNING_CURVE_POINTS, 'HOG Train/Test Accuracy')

print('\nExtracting SIFT descriptors...')
sift = cv2.SIFT_create()

##################### Extract SIFT descriptors ##################
def compute_sift_descriptors(images, max_descriptors=SIFT_MAX_DESCRIPTORS_PER_IMAGE):
    per_image = []
    for img in tqdm(images, desc='SIFT descriptors'):
        keypoints, descriptors = sift.detectAndCompute(img, None)
        if descriptors is None or len(descriptors) == 0:
            descriptors = np.zeros((1, 128), dtype=np.float32)
        descriptors = descriptors[:max_descriptors].astype(np.float32)
        per_image.append(descriptors)
    return per_image

train_sift_desc = compute_sift_descriptors(X_train_imgs)
test_sift_desc = compute_sift_descriptors(X_test_imgs)
all_train_descriptors = np.vstack(train_sift_desc)
codebook_size = min(SIFT_CODEBOOK_SIZE, max(8, len(all_train_descriptors) // 4))
kmeans = MiniBatchKMeans(n_clusters=codebook_size, random_state=SEED, batch_size=2048)
kmeans.fit(all_train_descriptors)

##################### Build bag-of-visual-words features ##################
def build_bovw_features(descriptor_list, model, vocab_size):
    features = np.zeros((len(descriptor_list), vocab_size), dtype=np.float32)
    for idx, descriptors in enumerate(descriptor_list):
        words = model.predict(descriptors)
        hist = np.bincount(words, minlength=vocab_size).astype(np.float32)
        features[idx] = hist / max(hist.sum(), 1.0)
    return features

X_train_sift = build_bovw_features(train_sift_desc, kmeans, codebook_size)
X_test_sift = build_bovw_features(test_sift_desc, kmeans, codebook_size)

sift_model = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=2500, multi_class='auto', n_jobs=None))
])
sift_model.fit(X_train_sift, y_train)
sift_pred = sift_model.predict(X_test_sift)
sift_metrics = compute_metrics(y_test, sift_pred)
plot_learning_curve_from_fractions(sift_model, X_train_sift, y_train, X_test_sift, y_test, LEARNING_CURVE_POINTS, 'SIFT Bag-of-Words Train/Test Accuracy')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('\nTraining CNN on', device)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

##################### Wrap signature data for PyTorch ##################
class SignatureDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform
    def __len__(self):
        return len(self.images)
    def __getitem__(self, idx):
        image = Image.fromarray(self.images[idx])
        if self.transform:
            image = self.transform(image)
        label = int(self.labels[idx])
        return image, label

train_loader = DataLoader(SignatureDataset(X_train_imgs, y_train, transform), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(SignatureDataset(X_test_imgs, y_test, transform), batch_size=BATCH_SIZE, shuffle=False)

##################### Define the CNN model ##################
class SmallSignatureCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4))
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

cnn_model = SmallSignatureCNN(num_classes=len(label_names)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(cnn_model.parameters(), lr=LEARNING_RATE)

history = {'train_loss': [], 'test_loss': [], 'train_acc': [], 'test_acc': []}
best_cnn_state = None
best_test_loss = float('inf')
best_cnn_epoch = 0
cnn_epochs_without_improvement = 0

##################### Evaluate the CNN ##################
def evaluate_torch_model(model, loader):
    model.eval()
    total_loss = 0.0
    preds, truths = [], []
    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            total_loss += loss.item() * inputs.size(0)
            preds.extend(outputs.argmax(dim=1).cpu().numpy())
            truths.extend(targets.cpu().numpy())
    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(truths, preds)
    return avg_loss, acc, np.array(truths), np.array(preds)

for epoch in range(1, CNN_EPOCHS + 1):
    cnn_model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for inputs, targets in tqdm(train_loader, desc=f'CNN epoch {epoch}/{CNN_EPOCHS}'):
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = cnn_model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        predictions = outputs.argmax(dim=1)
        correct += (predictions == targets).sum().item()
        total += targets.size(0)
    train_loss = running_loss / total
    train_acc = correct / total
    test_loss, test_acc, _, _ = evaluate_torch_model(cnn_model, test_loader)
    history['train_loss'].append(train_loss)
    history['test_loss'].append(test_loss)
    history['train_acc'].append(train_acc)
    history['test_acc'].append(test_acc)
    print(f'Epoch {epoch:02d} | train_loss={train_loss:.4f} | test_loss={test_loss:.4f} | train_acc={train_acc:.4f} | test_acc={test_acc:.4f}')
    if test_loss < best_test_loss - 1e-4:
        best_test_loss = test_loss
        best_cnn_epoch = epoch
        best_cnn_state = {k: v.detach().cpu().clone() for k, v in cnn_model.state_dict().items()}
        cnn_epochs_without_improvement = 0
    else:
        cnn_epochs_without_improvement += 1
        if cnn_epochs_without_improvement >= CNN_EARLY_STOPPING_PATIENCE:
            print(f'Early stopping triggered at epoch {epoch}. Best epoch was {best_cnn_epoch}.')
            break

if best_cnn_state is not None:
    cnn_model.load_state_dict(best_cnn_state)

_, _, y_test_torch, cnn_pred = evaluate_torch_model(cnn_model, test_loader)
cnn_metrics = compute_metrics(y_test_torch, cnn_pred)

epochs = np.arange(1, CNN_EPOCHS + 1)
plt.figure(figsize=(8, 4))
plt.plot(epochs, history['train_acc'], marker='o', label='Train accuracy')
plt.plot(epochs, history['test_acc'], marker='s', label='Test accuracy')
plt.title('CNN Train/Test Accuracy by Epoch')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim(0, 1.05)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(epochs, history['train_loss'], marker='o', label='Train error (loss)')
plt.plot(epochs, history['test_loss'], marker='s', label='Test error (loss)')
plt.title('CNN Train/Test Error by Epoch')
plt.xlabel('Epoch')
plt.ylabel('Cross-entropy loss')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

results = pd.DataFrame([
    {'model': 'HOG + Logistic Regression', **hog_metrics},
    {'model': 'SIFT BoVW + Logistic Regression', **sift_metrics},
    {'model': 'CNN', **cnn_metrics}
]).sort_values('f1_score', ascending=False).reset_index(drop=True)

display(results.style.format({col: '{:.4f}' for col in ['accuracy', 'precision', 'recall', 'f1_score']}))

results_melted = results.melt(id_vars='model', var_name='metric', value_name='score')
plt.figure(figsize=(10, 5))
sns.barplot(data=results_melted, x='metric', y='score', hue='model')
plt.ylim(0, 1.05)
plt.title('Performance Comparison Across Feature Extraction Techniques')
plt.tight_layout()
plt.show()

best_model_name = results.loc[0, 'model']
best_pred = {
    'HOG + Logistic Regression': hog_pred,
    'SIFT BoVW + Logistic Regression': sift_pred,
    'CNN': cnn_pred
}[best_model_name]

cm = confusion_matrix(y_test, best_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, cmap='Blues', cbar=False)
plt.title(f'Confusion Matrix for Best Model: {best_model_name}')
plt.xlabel('Predicted label index')
plt.ylabel('True label index')
plt.tight_layout()
plt.show()

print('\nClassification report for best model:')
print(classification_report(y_test, best_pred, target_names=label_names, zero_division=0))

best_row = results.iloc[0]
worst_row = results.iloc[-1]
print('Conclusion:')
print(
    f"The best-performing approach on this train/test split is {best_row['model']} with "
    f"accuracy={best_row['accuracy']:.4f}, precision={best_row['precision']:.4f}, "
    f"recall={best_row['recall']:.4f}, and F1-score={best_row['f1_score']:.4f}. "
    f"Compared with the weakest result ({worst_row['model']}), the best model improves F1-score by "
    f"{best_row['f1_score'] - worst_row['f1_score']:.4f}."
)
print(
    'Interpretation: if the CNN is best, learned features captured signer-specific stroke patterns more effectively than hand-crafted descriptors; '
    'if HOG or SIFT is best, the dataset size or signature consistency favored manual feature engineering.'
)

## Word-Level LSTM Sentence Completion




In [ ]:
import importlib.util
import math
import random
import re
import subprocess
import sys
from collections import Counter
from pathlib import Path

REQUIRED_PACKAGES = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'matplotlib': 'matplotlib',
    'torch': 'torch',
    'ipywidgets': 'ipywidgets'
}
missing = [pkg for mod, pkg in REQUIRED_PACKAGES.items() if importlib.util.find_spec(mod) is None]
if missing:
    print('Installing missing packages:', ', '.join(missing))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import HTML, display
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
import ipywidgets as widgets

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

TEXT_PATH = Path('./shakespeare-plays/versions/4/alllines.txt')
SEQ_LEN = 6
MIN_WORD_FREQ = 3
MAX_VOCAB_SIZE = 8000
MAX_TOKENS = 140000
TRAIN_RATIO = 0.9
EARLY_STOPPING_PATIENCE = 3
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if not TEXT_PATH.exists():
    raise FileNotFoundError(f'Shakespeare text not found at: {TEXT_PATH.resolve()}')

##################### Read raw text ##################
raw_lines = []
with TEXT_PATH.open('r', encoding='utf-8', errors='ignore') as f:
    for line in f:
        line = line.strip().strip('"')
        if line:
            raw_lines.append(line.lower())

# Keep apostrophes inside words and preserve basic punctuation as separate tokens.
##################### Split a line into tokens ##################
def tokenize_line(line):
    return re.findall(r"[a-z']+|[.,!?;:]", line)

all_tokens = []
for line in raw_lines:
    tokens = tokenize_line(line)
    if len(tokens) >= 3:
        all_tokens.extend(tokens + ['<eol>'])

if len(all_tokens) > MAX_TOKENS:
    all_tokens = all_tokens[:MAX_TOKENS]

##################### Build vocabulary ##################
counter = Counter(all_tokens)
base_vocab = ['<pad>', '<unk>', '<eol>']
most_common_words = [
    word for word, freq in counter.most_common(MAX_VOCAB_SIZE)
    if freq >= MIN_WORD_FREQ and word not in {'<pad>', '<unk>'}
]
vocab = base_vocab + [word for word in most_common_words if word not in base_vocab]
word_to_idx = {word: idx for idx, word in enumerate(vocab)}
idx_to_word = {idx: word for word, idx in word_to_idx.items()}
PAD_IDX = word_to_idx['<pad>']
UNK_IDX = word_to_idx['<unk>']

encoded_tokens = [word_to_idx.get(token, UNK_IDX) for token in all_tokens]
##################### Build training pairs ##################
pairs = []
for i in range(len(encoded_tokens) - SEQ_LEN):
    context = encoded_tokens[i:i + SEQ_LEN]
    target = encoded_tokens[i + SEQ_LEN]
    if PAD_IDX not in context:
        pairs.append((context, target))

split_idx = int(len(pairs) * TRAIN_RATIO)
train_pairs = pairs[:split_idx]
val_pairs = pairs[split_idx:]

##################### Wrap text pairs for PyTorch ##################
class ShakespeareDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples
    def __len__(self):
        return len(self.examples)
    def __getitem__(self, idx):
        x, y = self.examples[idx]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

train_dataset = ShakespeareDataset(train_pairs)
val_dataset = ShakespeareDataset(val_pairs)

##################### Define the LSTM model ##################
class NextWordLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, vocab_size)
    def forward(self, x):
        x = self.embedding(x)
        output, _ = self.lstm(x)
        last_hidden = output[:, -1, :]
        return self.fc(self.dropout(last_hidden))


##################### Create a data loader ##################
def make_loader(dataset, batch_size, shuffle):
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


##################### Evaluate one text model ##################
def evaluate_model(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            logits = model(xb)
            loss = criterion(logits, yb)
            total_loss += loss.item() * xb.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)
    avg_loss = total_loss / max(total, 1)
    avg_acc = correct / max(total, 1)
    ppl = math.exp(min(avg_loss, 20))
    return avg_loss, avg_acc, ppl


##################### Train one LSTM experiment ##################
def train_experiment(name, embed_dim, hidden_dim, num_layers, dropout, batch_size, epochs, lr):
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    train_loader = make_loader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = make_loader(val_dataset, batch_size=batch_size, shuffle=False)
    model = NextWordLSTM(len(vocab), embed_dim, hidden_dim, num_layers, dropout).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = {
        'epoch': [],
        'train_loss': [],
        'val_loss': [],
        'train_acc': [],
        'val_acc': [],
        'val_perplexity': []
    }
    best_state = None
    best_val_loss = float('inf')
    best_epoch = 0
    epochs_without_improvement = 0

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        running_correct = 0
        running_total = 0
        for xb, yb in tqdm(train_loader, desc=f'{name} | epoch {epoch}/{epochs}'):
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * xb.size(0)
            preds = logits.argmax(dim=1)
            running_correct += (preds == yb).sum().item()
            running_total += yb.size(0)

        train_loss = running_loss / max(running_total, 1)
        train_acc = running_correct / max(running_total, 1)
        val_loss, val_acc, val_ppl = evaluate_model(model, val_loader, criterion)

        history['epoch'].append(epoch)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['val_perplexity'].append(val_ppl)

        print(
            f"{name} | epoch={epoch:02d} | train_loss={train_loss:.4f} | "
            f"val_loss={val_loss:.4f} | train_acc={train_acc:.4f} | "
            f"val_acc={val_acc:.4f} | val_ppl={val_ppl:.2f}"
        )

        if val_loss < best_val_loss - 1e-4:
            best_val_loss = val_loss
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
                print(f'Early stopping triggered at epoch {epoch}. Best epoch was {best_epoch}.')
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    result = {
        'name': name,
        'config': {
            'embed_dim': embed_dim,
            'hidden_dim': hidden_dim,
            'num_layers': num_layers,
            'dropout': dropout,
            'batch_size': batch_size,
            'epochs': epochs,
            'patience': EARLY_STOPPING_PATIENCE,
            'lr': lr
        },
        'model': model,
        'history': history,
        'best_epoch': best_epoch,
        'best_val_loss': best_val_loss,
        'best_val_acc': max(history['val_acc']),
        'best_val_perplexity': min(history['val_perplexity'])
    }
    experiment_results[name] = result

    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(history['epoch'], history['train_loss'], marker='o', label='Train loss')
    plt.plot(history['epoch'], history['val_loss'], marker='s', label='Validation loss')
    plt.title(f'{name}: Train vs Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.grid(True, alpha=0.3)
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history['epoch'], history['train_acc'], marker='o', label='Train accuracy')
    plt.plot(history['epoch'], history['val_acc'], marker='s', label='Validation accuracy')
    plt.title(f'{name}: Train vs Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.ylim(0, 1.0)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    return result


##################### Convert prompt to model input ##################
def prompt_to_context(prompt, seq_len=SEQ_LEN):
    tokens = tokenize_line(prompt.lower())
    if not tokens:
        tokens = ['the']
    ids = [word_to_idx.get(tok, UNK_IDX) for tok in tokens][-seq_len:]
    if len(ids) < seq_len:
        ids = [PAD_IDX] * (seq_len - len(ids)) + ids
    return torch.tensor([ids], dtype=torch.long, device=DEVICE)


##################### Predict next-word choices ##################
def predict_next_words(model, prompt, top_k=5):
    model.eval()
    context = prompt_to_context(prompt)
    with torch.no_grad():
        probs = torch.softmax(model(context), dim=1).squeeze(0).cpu().numpy()
    ranked = probs.argsort()[::-1]
    suggestions = []
    for idx in ranked:
        token = idx_to_word[idx]
        if token in {'<pad>', '<unk>', '<eol>'}:
            continue
        suggestions.append((token, float(probs[idx])))
        if len(suggestions) == top_k:
            break
    return suggestions


##################### Generate a short continuation ##################
def generate_completion(model, prompt, steps=12):
    generated = tokenize_line(prompt.lower())
    for _ in range(steps):
        current_prompt = ' '.join(generated[-SEQ_LEN:]) if generated else prompt
        next_candidates = predict_next_words(model, current_prompt, top_k=1)
        if not next_candidates:
            break
        next_word = next_candidates[0][0]
        if next_word == '<eol>':
            break
        generated.append(next_word)
    return ' '.join(generated)


experiment_results = {}
BEST_EXPERIMENT_NAME = None
BEST_MODEL = None

print('Device:', DEVICE)
print('Text path:', TEXT_PATH.resolve())
print('Total raw lines:', len(raw_lines))
print('Total tokens used:', len(all_tokens))
print('Vocabulary size:', len(vocab))
print('Training sequences:', len(train_dataset))
print('Validation sequences:', len(val_dataset))
print('Example tokens:', all_tokens[:20])


In [ ]:
# Hyperparameter Experiment 1: compact baseline model
exp_baseline = train_experiment(
    name='Experiment 1 - Baseline',
    embed_dim=64,
    hidden_dim=128,
    num_layers=1,
    dropout=0.20,
    batch_size=128,
    epochs=50,
    lr=1e-3
)


In [ ]:
# Hyperparameter Experiment 2: larger embedding and deeper LSTM
exp_deeper = train_experiment(
    name='Experiment 2 - Deeper LSTM',
    embed_dim=128,
    hidden_dim=256,
    num_layers=2,
    dropout=0.30,
    batch_size=128,
    epochs=10,
    lr=1e-3
)


In [ ]:
# Hyperparameter Experiment 3: stronger regularization with smaller batch size
exp_regularized = train_experiment(
    name='Experiment 3 - Regularized',
    embed_dim=128,
    hidden_dim=192,
    num_layers=2,
    dropout=0.45,
    batch_size=96,
    epochs=10,
    lr=7e-4
)


In [ ]:
if not experiment_results:
    raise RuntimeError('Run the experiment cells before running the comparison cell.')

##################### Compare experiments ##################
summary_rows = []
for name, result in experiment_results.items():
    summary_rows.append({
        'experiment': name,
        'best_val_loss': result['best_val_loss'],
        'best_val_acc': result['best_val_acc'],
        'best_val_perplexity': result['best_val_perplexity'],
        **result['config']
    })

summary_df = pd.DataFrame(summary_rows).sort_values(['best_val_loss', 'best_val_perplexity'])
display(summary_df.style.format({
    'best_val_loss': '{:.4f}',
    'best_val_acc': '{:.4f}',
    'best_val_perplexity': '{:.2f}',
    'lr': '{:.5f}'
}))

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.bar(summary_df['experiment'], summary_df['best_val_acc'], color='#1f77b4')
plt.xticks(rotation=20, ha='right')
plt.ylim(0, 1.0)
plt.ylabel('Best validation accuracy')
plt.title('Hyperparameter Comparison: Accuracy')

plt.subplot(1, 2, 2)
plt.bar(summary_df['experiment'], summary_df['best_val_perplexity'], color='#d62728')
plt.xticks(rotation=20, ha='right')
plt.ylabel('Best validation perplexity')
plt.title('Hyperparameter Comparison: Perplexity')
plt.tight_layout()
plt.show()

##################### Select best model ##################
BEST_EXPERIMENT_NAME = summary_df.iloc[0]['experiment']
BEST_MODEL = experiment_results[BEST_EXPERIMENT_NAME]['model']
print('Best experiment selected for inference:', BEST_EXPERIMENT_NAME)

sample_prompts = [
    'to be or not',
    'my lord',
    'the king',
    'i will',
    'love is'
]

qualitative_rows = []
for prompt in sample_prompts:
    row = {'prompt': prompt}
    for name, result in experiment_results.items():
        completion = generate_completion(result['model'], prompt, steps=8)
        row[name] = completion
    qualitative_rows.append(row)

qualitative_df = pd.DataFrame(qualitative_rows)
display(qualitative_df)

print('Interpretation:')
print('- Lower validation loss and perplexity indicate more fluent and confident next-word predictions.')
print('- Higher validation accuracy indicates stronger exact next-word prediction performance.')
print('- Compare the generated completions above to judge coherence, repetition, and fluency qualitatively.')


In [ ]:
if BEST_MODEL is None:
    raise RuntimeError('Run the comparison cell first so the best model can be selected for the UI.')

prompt_box = widgets.Text(
    value='to be or not',
    description='Prompt:',
    placeholder='Type a partial sentence here',
    layout=widgets.Layout(width='90%')
)

next_word_slider = widgets.IntSlider(
    value=5,
    min=3,
    max=10,
    step=1,
    description='Top K:',
    continuous_update=False
)

continuation_slider = widgets.IntSlider(
    value=10,
    min=5,
    max=20,
    step=1,
    description='Steps:',
    continuous_update=False
)

output = widgets.HTML()


##################### Update the suggestion UI ##################
def render_suggestions(*_):
    prompt = prompt_box.value.strip() or 'the'
    top_k = next_word_slider.value
    steps = continuation_slider.value
    suggestions = predict_next_words(BEST_MODEL, prompt, top_k=top_k)
    completion = generate_completion(BEST_MODEL, prompt, steps=steps)
    rows = ''.join(
        f"<tr><td style='padding:4px 10px'>{i + 1}</td><td style='padding:4px 10px'><b>{word}</b></td><td style='padding:4px 10px'>{prob:.4f}</td></tr>"
        for i, (word, prob) in enumerate(suggestions)
    )
    output.value = f"""
    <div style='font-family:Arial, sans-serif'>
        <h3 style='margin-bottom:8px'>Real-Time Next Word Suggestions</h3>
        <p><b>Best model:</b> {BEST_EXPERIMENT_NAME}</p>
        <p><b>Greedy continuation:</b> {completion}</p>
        <table style='border-collapse:collapse; min-width:360px'>
            <tr>
                <th style='text-align:left; padding:4px 10px'>Rank</th>
                <th style='text-align:left; padding:4px 10px'>Suggested word</th>
                <th style='text-align:left; padding:4px 10px'>Probability</th>
            </tr>
            {rows}
        </table>
    </div>
    """

prompt_box.observe(render_suggestions, names='value')
next_word_slider.observe(render_suggestions, names='value')
continuation_slider.observe(render_suggestions, names='value')
render_suggestions()

display(widgets.VBox([prompt_box, widgets.HBox([next_word_slider, continuation_slider]), output]))


PixelCNN on Binarized MNIST

In [ ]:
import copy
import importlib.util
import struct
import subprocess
import sys
from pathlib import Path

PIXELCNN_REQUIRED_PACKAGES = {
    'numpy': 'numpy',
    'matplotlib': 'matplotlib',
    'torch': 'torch'
}
pixelcnn_missing = [pkg for mod, pkg in PIXELCNN_REQUIRED_PACKAGES.items() if importlib.util.find_spec(mod) is None]
if pixelcnn_missing:
    print('Installing missing packages:', ', '.join(pixelcnn_missing))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *pixelcnn_missing])

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

MNIST_DIR = Path('./mnist')
TRAIN_IMAGE_LIMIT = 12000
VAL_IMAGE_LIMIT = 2000
TEST_IMAGE_LIMIT = 2000
BINARY_THRESHOLD = 0.33
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

##################### Find IDX files ##################
def resolve_idx_file(*patterns):
    for pattern in patterns:
        matches = sorted(MNIST_DIR.glob(pattern))
        if matches:
            return matches[0]
    raise FileNotFoundError(f'Could not find any of these files under {MNIST_DIR.resolve()}: {patterns}')

train_images_path = resolve_idx_file('train-images.idx3-ubyte', 'train-images-idx3-ubyte/train-images-idx3-ubyte')
train_labels_path = resolve_idx_file('train-labels.idx1-ubyte', 'train-labels-idx1-ubyte/train-labels-idx1-ubyte')
test_images_path = resolve_idx_file('t10k-images.idx3-ubyte', 't10k-images-idx3-ubyte/t10k-images-idx3-ubyte')
test_labels_path = resolve_idx_file('t10k-labels.idx1-ubyte', 't10k-labels-idx1-ubyte/t10k-labels-idx1-ubyte')

##################### Read IDX images ##################
def read_idx_images(path):
    with open(path, 'rb') as f:
        magic, n_images, rows, cols = struct.unpack('>IIII', f.read(16))
        if magic != 2051:
            raise ValueError(f'Unexpected image magic number in {path}: {magic}')
        data = np.frombuffer(f.read(), dtype=np.uint8).reshape(n_images, rows, cols)
    return data.astype(np.float32) / 255.0

##################### Read IDX labels ##################
def read_idx_labels(path):
    with open(path, 'rb') as f:
        magic, n_items = struct.unpack('>II', f.read(8))
        if magic != 2049:
            raise ValueError(f'Unexpected label magic number in {path}: {magic}')
        data = np.frombuffer(f.read(), dtype=np.uint8)
    return data

##################### Split MNIST data ##################
train_images_full = read_idx_images(train_images_path)
train_labels_full = read_idx_labels(train_labels_path)
test_images_full = read_idx_images(test_images_path)
test_labels_full = read_idx_labels(test_labels_path)

train_images = train_images_full[:TRAIN_IMAGE_LIMIT]
train_labels = train_labels_full[:TRAIN_IMAGE_LIMIT]
val_images = train_images_full[TRAIN_IMAGE_LIMIT:TRAIN_IMAGE_LIMIT + VAL_IMAGE_LIMIT]
val_labels = train_labels_full[TRAIN_IMAGE_LIMIT:TRAIN_IMAGE_LIMIT + VAL_IMAGE_LIMIT]
test_images = test_images_full[:TEST_IMAGE_LIMIT]
test_labels = test_labels_full[:TEST_IMAGE_LIMIT]

##################### Binarize images ##################
train_images = (train_images > BINARY_THRESHOLD).astype(np.float32)
val_images = (val_images > BINARY_THRESHOLD).astype(np.float32)
test_images = (test_images > BINARY_THRESHOLD).astype(np.float32)

##################### Wrap MNIST for PyTorch ##################
class BinaryMNISTDataset(Dataset):
    def __init__(self, images, labels):
        self.images = torch.tensor(images[:, None, :, :], dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.images)
    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]

train_dataset = BinaryMNISTDataset(train_images, train_labels)
val_dataset = BinaryMNISTDataset(val_images, val_labels)
test_dataset = BinaryMNISTDataset(test_images, test_labels)

##################### Mask future pixels ##################
class MaskedConv2d(nn.Conv2d):
    def __init__(self, mask_type, *args, **kwargs):
        super().__init__(*args, **kwargs)
        if mask_type not in {'A', 'B'}:
            raise ValueError('mask_type must be A or B')
        self.mask_type = mask_type
        self.register_buffer('mask', torch.ones_like(self.weight))
        _, _, kH, kW = self.weight.shape
        self.mask[:, :, kH // 2, kW // 2 + (mask_type == 'B'):] = 0
        self.mask[:, :, kH // 2 + 1:] = 0
    def forward(self, x):
        return nn.functional.conv2d(
            x,
            self.weight * self.mask,
            self.bias,
            self.stride,
            self.padding,
            self.dilation,
            self.groups
        )

##################### Define the PixelCNN model ##################
class PixelCNN(nn.Module):
    def __init__(self, hidden_channels=64, n_layers=6, dropout=0.2):
        super().__init__()
        layers = [
            MaskedConv2d('A', 1, hidden_channels, kernel_size=7, padding=3),
            nn.ReLU()
        ]
        for _ in range(n_layers):
            layers.extend([
                MaskedConv2d('B', hidden_channels, hidden_channels, kernel_size=3, padding=1),
                nn.BatchNorm2d(hidden_channels),
                nn.ReLU(),
                nn.Dropout2d(dropout)
            ])
        layers.extend([
            MaskedConv2d('B', hidden_channels, hidden_channels, kernel_size=1),
            nn.ReLU(),
            MaskedConv2d('B', hidden_channels, 1, kernel_size=1)
        ])
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)


##################### Create MNIST loaders ##################
def make_loaders(batch_size):
    return (
        DataLoader(train_dataset, batch_size=batch_size, shuffle=True),
        DataLoader(val_dataset, batch_size=batch_size, shuffle=False),
        DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    )


##################### Evaluate PixelCNN ##################
def evaluate_pixelcnn(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    total_count = 0
    with torch.no_grad():
        for xb, _ in loader:
            xb = xb.to(DEVICE)
            logits = model(xb)
            loss = criterion(logits, xb)
            total_loss += loss.item() * xb.size(0)
            total_count += xb.size(0)
    return total_loss / max(total_count, 1)


##################### Train one PixelCNN experiment ##################
def train_pixelcnn_experiment(name, hidden_channels, n_layers, dropout, batch_size, lr, max_epochs, patience):
    model = PixelCNN(hidden_channels=hidden_channels, n_layers=n_layers, dropout=dropout).to(DEVICE)
    train_loader, val_loader, test_loader = make_loaders(batch_size)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = {
        'epoch': [],
        'train_nll': [],
        'val_nll': []
    }
    best_state = None
    best_val_loss = float('inf')
    best_epoch = 0
    epochs_without_improvement = 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        running_loss = 0.0
        sample_count = 0
        for xb, _ in tqdm(train_loader, desc=f'{name} | epoch {epoch}/{max_epochs}'):
            xb = xb.to(DEVICE)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, xb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * xb.size(0)
            sample_count += xb.size(0)

        train_nll = running_loss / max(sample_count, 1)
        val_nll = evaluate_pixelcnn(model, val_loader, criterion)
        history['epoch'].append(epoch)
        history['train_nll'].append(train_nll)
        history['val_nll'].append(val_nll)

        print(f'{name} | epoch={epoch:02d} | train_nll={train_nll:.4f} | val_nll={val_nll:.4f}')

        if val_nll < best_val_loss - 1e-4:
            best_val_loss = val_nll
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f'Early stopping triggered at epoch {epoch}. Best epoch was {best_epoch}.')
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    test_nll = evaluate_pixelcnn(model, test_loader, criterion)
    result = {
        'name': name,
        'config': {
            'hidden_channels': hidden_channels,
            'n_layers': n_layers,
            'dropout': dropout,
            'batch_size': batch_size,
            'lr': lr,
            'max_epochs': max_epochs,
            'patience': patience
        },
        'model': model,
        'history': history,
        'best_val_nll': best_val_loss,
        'best_epoch': best_epoch,
        'test_nll': test_nll
    }
    pixelcnn_results[name] = result

    plt.figure(figsize=(8, 4))
    plt.plot(history['epoch'], history['train_nll'], marker='o', label='Train NLL')
    plt.plot(history['epoch'], history['val_nll'], marker='s', label='Validation NLL')
    plt.axvline(best_epoch, color='gray', linestyle='--', alpha=0.7, label=f'Best epoch: {best_epoch}')
    plt.title(f'{name}: Train vs Validation NLL')
    plt.xlabel('Epoch')
    plt.ylabel('Negative log-likelihood (Bernoulli NLL)')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    return result


##################### Sample new images ##################
def sample_from_pixelcnn(model, n_samples=16):
    model.eval()
    samples = torch.zeros(n_samples, 1, 28, 28, device=DEVICE)
    with torch.no_grad():
        for row in range(28):
            for col in range(28):
                logits = model(samples)
                probs = torch.sigmoid(logits[:, :, row, col])
                samples[:, :, row, col] = torch.bernoulli(probs)
    return samples.cpu().numpy().squeeze(1)


##################### Display image grid ##################
def show_image_grid(images, title, ncols=4):
    n_images = len(images)
    nrows = int(np.ceil(n_images / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 2, nrows * 2))
    axes = np.array(axes).reshape(-1)
    for ax, image in zip(axes, images):
        ax.imshow(image, cmap='gray')
        ax.axis('off')
    for ax in axes[n_images:]:
        ax.axis('off')
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

pixelcnn_results = {}
BEST_PIXELCNN_NAME = None
BEST_PIXELCNN_MODEL = None

print('Device:', DEVICE)
print('Train/Val/Test sizes:', len(train_dataset), len(val_dataset), len(test_dataset))
print('Using dataset files:')
print(' ', train_images_path)
print(' ', train_labels_path)
print(' ', test_images_path)
print(' ', test_labels_path)
show_image_grid(train_images[:16], 'Binarized MNIST Training Samples')


In [ ]:
# PixelCNN Experiment 1: baseline PixelCNN
q3_exp_baseline = train_pixelcnn_experiment(
    name='PixelCNN Experiment 1 - Baseline',
    hidden_channels=48,
    n_layers=4,
    dropout=0.10,
    batch_size=128,
    lr=1e-3,
    max_epochs=12,
    patience=3
)


In [ ]:
# PixelCNN Experiment 2: deeper PixelCNN with more channels
q3_exp_deeper = train_pixelcnn_experiment(
    name='PixelCNN Experiment 2 - Deeper',
    hidden_channels=64,
    n_layers=6,
    dropout=0.20,
    batch_size=128,
    lr=8e-4,
    max_epochs=14,
    patience=3
)


In [ ]:
# PixelCNN Experiment 3: stronger regularization and smaller batch size
q3_exp_regularized = train_pixelcnn_experiment(
    name='PixelCNN Experiment 3 - Regularized',
    hidden_channels=72,
    n_layers=5,
    dropout=0.30,
    batch_size=96,
    lr=6e-4,
    max_epochs=14,
    patience=4
)


In [ ]:
import pandas as pd
if not pixelcnn_results:
    raise RuntimeError('Run the PixelCNN experiment cells before the comparison cell.')

##################### Compare PixelCNN runs ##################
summary_rows = []
for name, result in pixelcnn_results.items():
    sample_images = sample_from_pixelcnn(result['model'], n_samples=16)
    pixel_density = float(sample_images.mean())
    summary_rows.append({
        'experiment': name,
        'best_val_nll': result['best_val_nll'],
        'best_epoch': result['best_epoch'],
        'test_nll': result['test_nll'],
        'generated_pixel_density': pixel_density,
        **result['config']
    })
    show_image_grid(sample_images, f'{name}: Generated MNIST Samples')

summary_df = pd.DataFrame(summary_rows).sort_values('best_val_nll').round(4)
display(summary_df)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.bar(summary_df['experiment'], summary_df['best_val_nll'], color='#1f77b4')
plt.xticks(rotation=20, ha='right')
plt.ylabel('Best validation NLL')
plt.title('PixelCNN Hyperparameter Comparison')

plt.subplot(1, 2, 2)
plt.bar(summary_df['experiment'], summary_df['test_nll'], color='#d62728')
plt.xticks(rotation=20, ha='right')
plt.ylabel('Test NLL')
plt.title('Generalization Comparison')
plt.tight_layout()
plt.show()

BEST_PIXELCNN_NAME = summary_df.iloc[0]['experiment']
BEST_PIXELCNN_MODEL = pixelcnn_results[BEST_PIXELCNN_NAME]['model']
print('Best experiment:', BEST_PIXELCNN_NAME)
print('Evaluation guidance:')
print('- Lower validation/test NLL means the model modeled the pixel distribution more accurately.')
print('- Generated images should look digit-like, structurally coherent, and not collapse to noise or a single repeated pattern.')
print('- Pixel density close to the MNIST training distribution is a useful sanity check for fluency of image generation.')
print('Reference training-set pixel density:', float(train_images.mean()))
